In [8]:
!pip install -q langchain langchain-groq

In [9]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)
print("Groq setup completed!")

Groq setup completed!


In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
- Plum cake: Rs 500
- Chocolate cake: Rs 650
- Cupcake: Rs 80 each
- Orders need 24 hours' notice.
- Eggless versions are available.

RULES:
- Never promise a delivery time or place.
- If you don't know something, say "I'll check with the owner."
- Calculate totals carefully.
- Keep replies under 50 words.
- Reply in English or Telugu, matching the customer.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()

history = []

print("Sweet Crumbs assistant is ready!")


Sweet Crumbs assistant is ready!


In [11]:
def ask(question: str) -> str:
    answer = chain.invoke({
      "history": history,
      "question": question
    })
    history.extend([
        ("human", question),
        ("ai", answer)
    ])
    return answer
print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What is my name?"))

Hi Ravi! Yes, we offer eggless versions of all our cakes, including plum, chocolate, and cupcakes. Let us know which one you’d like, and we’ll take care of the rest.
Your name is Ravi.


In [12]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv


In [17]:
import csv

def run_tests(path="/content/test.csv"):
    with open(path, "r", encoding="utf-8-sig", newline="") as file:
       rows = list(csv.DictReader(file.readlines()[1:], skipinitialspace=True))
    passed = 0
    for row in rows:
      answer = chain.invoke({
          "history": [],
          "question": row["question"]
      })
      expected = row["must_include"]
      ok = expected.lower() in answer.lower()
      if ok:
        passed +=1
      print("PASS" if ok else "FAIL", "|", row["question"])
      print("Expected:", expected)
      print("Answer:", answer)
      print("-" * 50)
    print(f"\n{passed}/{len(rows)} tests passed")

run_tests()

PASS | How much do 4 cupcakes cost?
Expected: 320
Answer: 4 cupcakes will cost **Rs 320**.
--------------------------------------------------
FAIL | What is the price of 3 plum cakes?
Expected: 1500
Answer: Three plum cakes cost Rs 1,500.
--------------------------------------------------
FAIL | How much for 2 chocolate cakes?
Expected: 1300
Answer: Two chocolate cakes cost ₹1,300. Eggless versions are also available, and we need a 24‑hour notice for orders. Let me know if you'd like to place an order!
--------------------------------------------------
PASS | What is the total for 1 plum cake and 4 cupcakes?
Expected: 820
Answer: The total is Rs 820 (Plum cake Rs 500 + 4 cupcakes Rs 80 each = Rs 320).
--------------------------------------------------
FAIL | What is the total for 2 chocolate cakes and 3 cupcakes?
Expected: 1540
Answer: 2 chocolate cakes (₹650 each) + 3 cupcakes (₹80 each) = ₹1,540. Please note orders need 24‑hour notice; eggless versions are available.
----------------

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)
print("Sweet Crumbs agent created!")

Sweet Crumbs agent created!


In [19]:
result = agent.invoke({
    "messages": [
       {"role": "user", "content": "Do you make eggless cakes?"}
    ]
})
print(result["messages"][-1].content)


Yes, we offer eggless versions of all our cakes, including plum, chocolate, and cupcakes. Let us know which one you’d like, and we’ll take care of the rest!
